# Baseline + ewaluacja — krok po kroku

Najnaiwniejszy sposób: **wszystkie akta w jeden prompt** + polecenie „jesteś prawnikiem, napisz apelację". Bez planu, bez podziału na kroki, bez selekcji dokumentów.

Ten notebook generuje apelację baseline, a potem **krok po kroku** ją ewaluuje: pokrycie (z powtarzalnością i kosztem) oraz halucynacje. Wynik jest punktem odniesienia dla agenta liniowego (`linear_walkthrough.ipynb`) i planera (`planner_walkthrough.ipynb`).

> Odpowiednik `baseline/main.py`, ale interaktywnie.

## 0. Konfiguracja

Konfiguracja LLM (klucz, model) jest w pliku `.env`. **Baseline wymaga modelu
z dużym oknem kontekstu** (prompt ~19k tokenów) — użyj OpenAI; lokalna Ollama
domyślnie obetnie kontekst do 4096.

In [7]:
import os

# Wczytaj konfigurację LLM z pliku .env (klucz nie jest wpisywany w komórce).
from dotenv import load_dotenv

load_dotenv()

print("model:", os.environ.get("LLM_MODEL"))
print("klucz z .env wczytany:", bool(os.environ.get("LLM_API_KEY")))

model: gpt-5.4-mini
klucz z .env wczytany: True


## 1. Wczytanie akt i rozmiar promptu

Baseline wrzuca **całość** w jeden prompt. Policzmy, ile to tokenów — to sedno problemu tego podejścia.

In [8]:
from src.loader import load_all
from src.tokens import count_tokens
from baseline.main import build_context
from baseline.prompts import SYSTEM_PROMPT

docs = load_all()
context = build_context(docs)
prompt_tokens = count_tokens(SYSTEM_PROMPT + "\n\n" + context)

print(f"Dokumentów: {len(docs)}")
print(f"Znaków w kontekście: {len(context):,}")
print(f"Tokenów w promptcie (wejście): ~{prompt_tokens:,}")

Dokumentów: 16
Znaków w kontekście: 52,159
Tokenów w promptcie (wejście): ~19,341


## 2. Generowanie apelacji (jeden prompt)

Jedno wywołanie LLM — całe akta wchodzą, wychodzi gotowa apelacja.

In [9]:
from baseline.main import generate_appeal
from src.llm import track_usage
from src.cost import estimate_cost

with track_usage() as u:
    appeal = generate_appeal(docs).tekst

cost = estimate_cost(u, os.environ["LLM_MODEL"])
print(f"Apelacja: {len(appeal):,} znaków")
print(
    f"Koszt generacji: ~${cost:.4f}  "
    f"({u.prompt_tokens:,} wej + {u.completion_tokens:,} wyj tok, {u.calls} wywołanie)\n"
)

Apelacja: 5,092 znaków
Koszt generacji: ~$0.0229  (19,473 wej + 1,851 wyj tok, 1 wywołanie)



In [12]:
from src.output import save_appeal

path = save_appeal(appeal, "baseline")
print("Zapisano apelację do:", path)

Zapisano apelację do: /Users/ewasuknarowska/Projects/WomenInTech/data/output/apelacja_baseline_2026-06-06_155720.txt


## (Opcjonalnie) Wczytaj zapisaną apelację

Zamiast generować od nowa (sekcja 2 — płatne wywołanie LLM), wczytaj **ostatnią**
zapisaną apelację z `data/output/`. Potem możesz puścić same sekcje 3–4 (ewaluacja).

> Do halucynacji (sekcja 4) potrzebny jest jeszcze `docs` — odpal sekcję 1
> (`load_all`, bez kosztu). Do samego pokrycia (sekcja 3) wystarczy `appeal`.

In [3]:
from src.output import load_latest_appeal, latest_appeal_path

appeal = load_latest_appeal("baseline")
print("Wczytano apelację z:", latest_appeal_path("baseline"))
print(f"{len(appeal):,} znaków")

Wczytano apelację z: /Users/ewasuknarowska/Projects/WomenInTech/data/output/apelacja_baseline_2026-06-06_155720.txt
7,911 znaków


## 3. Pokrycie — czy porusza wymagane zagadnienia?

Sędzia-LLM ocenia względem `data/eval.json`.

In [4]:
from src.eval.coverage import evaluate

cov = evaluate(appeal, print_results=True)

[1/12] ✅ 1. W zakresie czynu z art. 178a § 1 k.k. (zarzut pkt I z aktu oskarżenia) istotnym uchybieniem, jaki
     → Apelacja wyraźnie podnosi zarzut błędu w ustaleniach faktycznych co do czynu z art. 178a § 1 k.k., wskazując, że oskarżony miał być nietrzeźwy już o godz. 8.30 przy przyjeździe na cmentarz, mimo braku dowodu na tę okoliczność. Wprost akcentuje też brak przyznania się oskarżonego do tej okoliczności oraz brak podstaw dowodowych dla takiego ustalenia. Zagadnienie z klucza zostało więc faktycznie poruszone.

[2/12] ✅ 1. W zakresie czynu z art. 178a § 1 k.k. (zarzut pkt I z aktu oskarżenia) istotnym uchybieniem, jaki
     → Apelacja wyraźnie podnosi zarzut błędu w ustaleniach faktycznych co do czynu z art. 178a § 1 k.k., wskazując, że oskarżony miał być nietrzeźwy już przy przyjeździe na cmentarz bez podstawy dowodowej. Wprost przywołuje też zeznania świadka Karola Kota, podnosząc, że nie widział on momentu wjazdu ani sposobu znalezienia się pojazdu w alejce cmentarnej. To o

## 3b. Powtarzalność oceny + koszt

Uruchamiamy ocenę pokrycia **5 razy** na tej samej apelacji — czy sędzia jest
powtarzalny (temperatura 0, ale LLM bywa zmienny)? Przy okazji liczymy **koszt**
(tokeny × cennik) i koszt pojedynczego wywołania.

In [5]:
import statistics

from src.eval.coverage import evaluate
from src.llm import track_usage
from src.cost import estimate_cost

model = os.environ["LLM_MODEL"]
scores, costs, calls = [], [], 0
for i in range(1, 6):
    with track_usage() as u:
        cov = evaluate(appeal)  # 5 przebiegów na tej samej apelacji (bez druku)
    scores.append(cov.score)
    costs.append(estimate_cost(u, model))
    calls = u.calls
    print(
        f"Przebieg {i}: {cov.covered}/{cov.total} = {cov.score:.0%} "
        f"| {u.calls} wywołań | {u.total_tokens:,} tok | ${costs[-1]:.4f}"
    )

avg = statistics.mean(costs)
print(
    f"\nPokrycie — średnia {statistics.mean(scores):.0%} "
    f"(min {min(scores):.0%}, max {max(scores):.0%}, odch. {statistics.pstdev(scores):.1%})"
)
print(f"Koszt 1 przebiegu (~{calls} wywołań): ~${avg:.4f}")
print(f"Koszt 1 wywołania: ~${avg / calls:.5f}" if calls else "brak wywołań")

Przebieg 1: 6/12 = 50% | 12 wywołań | 41,070 tok | $0.0385
Przebieg 2: 5/12 = 42% | 12 wywołań | 41,120 tok | $0.0387
Przebieg 3: 6/12 = 50% | 12 wywołań | 41,137 tok | $0.0388
Przebieg 4: 6/12 = 50% | 12 wywołań | 41,113 tok | $0.0387
Przebieg 5: 5/12 = 42% | 12 wywołań | 41,041 tok | $0.0384

Pokrycie — średnia 47% (min 42%, max 50%, odch. 4.1%)
Koszt 1 przebiegu (~12 wywołań): ~$0.0386
Koszt 1 wywołania: ~$0.00322


## 4. Halucynacje — czy apelacja nie zmyśla faktów? (etapowo)

Zamiast jednym wywołaniem (`evaluate_grounding`), pokazujemy to **krok po kroku**:

1. **twierdzenia** wyciągnięte z apelacji,
2. **wybór dokumentów** do sprawdzenia każdego twierdzenia (po opisach akt),
3. **weryfikacja** — czy twierdzenie ma oparcie w wybranych dokumentach.

> Wymaga `docs` z sekcji 1 (`load_all`).

In [4]:
# Krok 1 — twierdzenia faktyczne wyciągnięte z apelacji
from src.eval.grounding import extract_claims

claims = extract_claims(appeal)
print(f"Wyciągnięto {len(claims)} twierdzeń:\n")
for i, c in enumerate(claims, 1):
    print(f"{i:2}. {c.claim}")

Wyciągnięto 30 twierdzeń:

 1. Wyrok Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie został wydany 14 marca 2025 r. w sprawie o sygnaturze II K 25/25.
 2. Oskarżonym w sprawie jest Daniel Dzik.
 3. Obrońcą Daniela Dzika jest radca prawny Lidia Lis.
 4. Apelacja została wniesiona od wyroku w całości, w zakresie punktów 1–9.
 5. Daniel Dzik złożył wyjaśnienia na rozprawie 27 lutego 2025 r.
 6. Daniel Dzik podał na rozprawie 27 lutego 2025 r., że przed przyjazdem na cmentarz był trzeźwy.
 7. Daniel Dzik podał na rozprawie 27 lutego 2025 r., że alkohol spożywał dopiero na cmentarzu w alejce.
 8. Świadek Karol Kot nie widział, kiedy Daniel Dzik wjechał na teren cmentarza.
 9. Świadek Karol Kot nie widział, w jaki sposób Daniel Dzik wjechał na teren cmentarza.
10. Świadek Karol Kot nie widział, aby Daniel Dzik odjeżdżał.
11. Parafia Rzymskokatolicka pw. Św. Krzysztofa w Szczyglicach odpowiedziała pismem z 1 marca 2025 r.
12. Brama zachodnia cmentarza pozostaje stale zamknięta.
13. Przez bra

### Krok 2 — wybór dokumentów do każdego twierdzenia

LLM na podstawie **opisów** akt (`file_description`) wskazuje, w których plikach
sprawdzić dany fakt — bez wczytywania wszystkich akt naraz.

In [5]:
from src.descriptions import get_descriptions
from src.eval.grounding import select_sources
from src.loader import load_all

# Akta: jeśli pominięto sekcję 1 (np. po restarcie kernela) — wczytaj je tutaj.
if "docs" not in globals():
    docs = load_all()

# Opisy z cache: liczone raz i współdzielone z agentami (data/output/described.json).
described = get_descriptions(docs)

# Wybór plików per twierdzenie — druk na bieżąco (każdy wybór trochę trwa).
selections = []
for i, claim in enumerate(claims, 1):
    sel = select_sources(claim.claim, described)
    selections.append((claim, sel))
    print(f"[{i}/{len(claims)}] {claim.claim}")
    print(f"   → wybrane pliki: {sel.files}\n", flush=True)

[1/30] Wyrok Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie został wydany 14 marca 2025 r. w sprawie o sygnaturze II K 25/25.
   → wybrane pliki: ['16_Wyrok.pdf', '15_Protokół_rozprawy_głównej.pdf']

[2/30] Oskarżonym w sprawie jest Daniel Dzik.
   → wybrane pliki: ['11_Akt_oskarżenia.pdf', '08_Postanowienie_o_przedstawieniu_zarzutów.pdf', '16_Wyrok.pdf']

[3/30] Obrońcą Daniela Dzika jest radca prawny Lidia Lis.
   → wybrane pliki: ['17_Wniosek_o_sporządzenie_uzasadnienia.pdf', '15_Protokół_rozprawy_głównej.pdf', '10_Protokół_przesłuchania_podejrzanego.pdf']

[4/30] Apelacja została wniesiona od wyroku w całości, w zakresie punktów 1–9.
   → wybrane pliki: ['17_Wniosek_o_sporządzenie_uzasadnienia.pdf', '15_Protokół_rozprawy_głównej.pdf']

[5/30] Daniel Dzik złożył wyjaśnienia na rozprawie 27 lutego 2025 r.
   → wybrane pliki: ['12_Protokół_rozprawy_głównej.pdf']

[6/30] Daniel Dzik podał na rozprawie 27 lutego 2025 r., że przed przyjazdem na cmentarz był trzeźwy.
   → wybrane pliki:

### Krok 3 — weryfikacja: czy twierdzenie to halucynacja?

Każde twierdzenie sprawdzamy **tylko** w wybranych dokumentach:
`supported` (potwierdzone) / `unsupported` (brak oparcia) / `contradicted` (sprzeczne).

In [6]:
from collections import Counter

from src.eval.grounding import verify_claim
from src.sources import prepare_input_texts

ICON = {"supported": "✅", "unsupported": "⚠️", "contradicted": "❌"}
statuses = []
for claim, sel in selections:
    sources_text = prepare_input_texts(docs, sel.files)
    v = verify_claim(claim.claim, sources_text)
    statuses.append(v.status)
    print(f"{ICON.get(v.status, '?')} [{v.status}] {claim.claim}")
    print(f"   → {v.reasoning}\n")

counts = Counter(statuses)
n = len(statuses)
rate = (counts["unsupported"] + counts["contradicted"]) / n if n else 0
print(f"Podsumowanie: {dict(counts)} | wskaźnik halucynacji: {rate:.0%}")

✅ [supported] Wyrok Sądu Rejonowego dla Krakowa–Krowodrzy w Krakowie został wydany 14 marca 2025 r. w sprawie o sygnaturze II K 25/25.
   → Akta wprost potwierdzają, że wyrok został wydany 14 marca 2025 r. przez Sąd Rejonowy dla Krakowa–Krowodrzy w Krakowie w sprawie o sygn. II K 25/25.

✅ [supported] Oskarżonym w sprawie jest Daniel Dzik.
   → Akta wprost wskazują, że sprawa dotyczy Daniela Dzika jako oskarżonego. W akcie oskarżenia jest zapis „przeciwko Danielowi Dzikowi”, a w wyroku „sprawy Daniela Dzika, ... oskarżonego”.

❌ [contradicted] Obrońcą Daniela Dzika jest radca prawny Lidia Lis.
   → Akta wprost wskazują, że obrońcą Daniela Dzika była radca prawny Gabriela Gil, a nie Lidia Lis. Lidia Lis występuje dopiero we wniosku o sporządzenie uzasadnienia jako radca prawny działający imieniem oskarżonego, z załączonym upoważnieniem do obrony, ale nie jako obrońca wprost wskazany w protokole przesłuchania czy rozprawy.

⚠️ [unsupported] Apelacja została wniesiona od wyroku w całości,

## Podsumowanie

Baseline = **1 wywołanie LLM** z ogromnym promptem (patrz tokeny w sekcji 1). Szybki do napisania, ale: zapchany kontekst, „lost in the middle", brak śladu rozumowania i podatność na pominięcia/halucynacje.